In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv("cardio.csv")

df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,bmi,bp_category,bp_category_encoded
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50,21.967120,Hypertension Stage 1,Hypertension Stage 1
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55,34.927679,Hypertension Stage 2,Hypertension Stage 2
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1,51,23.507805,Hypertension Stage 1,Hypertension Stage 1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48,28.710479,Hypertension Stage 2,Hypertension Stage 2
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0,47,23.011177,Normal,Normal


In [3]:
df.isnull().sum()

id                     0
age                    0
gender                 0
height                 0
weight                 0
ap_hi                  0
ap_lo                  0
cholesterol            0
gluc                   0
smoke                  0
alco                   0
active                 0
cardio                 0
age_years              0
bmi                    0
bp_category            0
bp_category_encoded    0
dtype: int64

In [4]:
features = [
    'age',
    'gender',
    'height',
    'weight',
    'bmi',
    'ap_hi',
    'ap_lo',
    'cholesterol',
    'gluc',
    'smoke',
    'alco',
    'active'
]

X = df[features]

In [5]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
wcss = []
silhouette_scores = []
db_scores = []

K = range(2,11)

for k in K:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=100
    )

    labels = kmeans.fit_predict(X_scaled)

    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

In [ ]:
print(f"k, Silhouette, Davies-Bouldin")

for i in range(len(K)):
    print(f"{K[i]},{wcss[i]:.2f}, {silhouette_scores[i]:.4f}, {db_scores[i]:.4f}")

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(K, wcss, marker='o')

plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(K, silhouette_scores, marker='o')

plt.title("Silhouette Score")
plt.xlabel("Number of Clusters")
plt.ylabel("Score")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(K, db_scores, marker='o')

plt.title("Davies-Bouldin Index")
plt.xlabel("Number of Clusters")
plt.ylabel("Score")

plt.grid(True)

plt.show()

In [ ]:
best_k = K[silhouette_scores.index(max(silhouette_scores))]

print("Best k =", best_k)

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

df.head()

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df["Cluster"],
    cmap="viridis",
    s=30
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Patient Clusters")

plt.colorbar(label="Cluster")

plt.show()

In [ ]:
centers = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df["Cluster"],
    cmap="viridis",
    s=25
)

plt.scatter(
    centers[:,0],
    centers[:,1],
    marker='X',
    s=250,
    c='red',
    label='Centroids'
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Clusters with Centroids")

plt.legend()

plt.show()

In [ ]:
cluster_summary = df.groupby("Cluster")[features].mean()

cluster_summary

In [ ]:
df["Cluster"].value_counts()

In [ ]:
df.groupby("Cluster")["cardio"].mean()*100

In [ ]:
new_patient = [[
    18393,
    2,
    170,
    75,
    25.95,
    130,
    85,
    2,
    1,
    0,
    0,
    1
]]

new_patient_scaled = scaler.transform(new_patient)

cluster = kmeans.predict(new_patient_scaled)

print("Patient belongs to Cluster:", cluster[0])